# FREYA: fine-tune on Google Colab

Runs top to bottom on a **free T4 GPU** (Runtime > Change runtime type > T4 GPU), then **Runtime > Run all**.

What it does:
1. Gets your FREYA code from GitHub and installs the training tools.
2. Builds the training data from `training/freya-examples.json`, plus any conversations you upload.
3. Checks every conversation against the limits below. A rejected file stops the run and lists the lines.
4. Trains a LoRA adapter on an uncensored Qwen3.5 4B and exports one GGUF file.
5. Loads it into Ollama **here on Colab** and runs the same 17-case persona evaluation as the laptop, so you can judge it before downloading 2-3 GB.
6. Saves it to Google Drive.

**Limits, always enforced on all data:** adults only, so nothing sexual or romantic involving minors; no non-consent; no examples teaching FREYA to claim she's human when someone sincerely asks. Explicit adult content in your own data is otherwise allowed.

Takes about 15-30 minutes, mostly downloads and training.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/KrishAnjara/freya-v1.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Qwen3.5 4B matches the laptop's model. The Qwen3 4B option is a text-only fallback if Qwen3.5 fails to load.
BASE_MODEL = "huihui-ai/Huihui-Qwen3.5-4B-abliterated"  #@param ["huihui-ai/Huihui-Qwen3.5-4B-abliterated", "huihui-ai/Huihui-Qwen3-4B-Instruct-2507-abliterated"]
EPOCHS = 2  #@param {type:"number"}
#@markdown Quick 10-step check that everything loads, with no export. Turn it off for the real run.
SMOKE_TEST = False  #@param {type:"boolean"}
UPLOAD_OWN_DATA = False  #@param {type:"boolean"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

In [ ]:
#@title 2. GPU check and code
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", gpu.stdout.strip())
if os.path.isdir("/content/freya-v1/.git"):
    !git -C /content/freya-v1 fetch -q origin {BRANCH} && git -C /content/freya-v1 checkout -q {BRANCH} && git -C /content/freya-v1 pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} /content/freya-v1
%cd /content/freya-v1
!git log --oneline -1

In [ ]:
#@title 3. Install training tools (and Node.js for FREYA's scripts)
!pip install -q unsloth trl datasets
!node --version 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null)
!node --version

In [ ]:
#@title 4. Your own conversations (optional)
#@markdown One JSON object per line: {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}. FREYA's system prompt is added automatically.
os.makedirs("training/data", exist_ok=True)
if UPLOAD_OWN_DATA:
    from google.colab import files
    for name, content in files.upload().items():
        with open(f"training/data/{os.path.basename(name)}", "wb") as handle:
            handle.write(content)
!ls -la training/data

In [ ]:
#@title 5. Build and validate the training data
!npx -y tsx scripts/build-training-data.mjs
!python training/train_lora.py --data training/freya-train.jsonl "training/data/*.jsonl" --validate-only

In [ ]:
#@title 6. Train
extra = "--max-steps 10 --no-gguf" if SMOKE_TEST else ""
%cd /content/freya-v1/training
!python train_lora.py --data freya-train.jsonl "data/*.jsonl" --base {BASE_MODEL} --epochs {EPOCHS} --out freya-gguf {extra}
%cd /content/freya-v1
!ls -lh training/freya-gguf

In [ ]:
#@title 7. Load into Ollama here and evaluate
import glob, time
assert not SMOKE_TEST, "Smoke test skips the export. Turn SMOKE_TEST off and run cell 6 again."
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
gguf = sorted((p for p in glob.glob("training/freya-gguf/*.gguf") if "mmproj" not in p.lower()), key=os.path.getsize)[-1]
# Qwen3.5 uses Ollama's built-in Qwen3.5 chat formatting; the Qwen3 fallback uses ChatML.
CHAT_FORMAT = "qwen3.5" if "Qwen3.5" in BASE_MODEL else "chatml"
print("Model file:", gguf, "| chat format:", CHAT_FORMAT)
!npx -y tsx scripts/ollama-create.mjs --base {gguf} --name freya-trained --format {CHAT_FORMAT}
!npx -y tsx scripts/eval-persona.mjs freya-trained

In [ ]:
#@title 8. Save to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/freya
    !cp {gguf} /content/drive/MyDrive/freya/freya-trained.gguf
    !cp -r training/freya-gguf/lora /content/drive/MyDrive/freya/
    print("Saved: Google Drive > MyDrive > freya > freya-trained.gguf")

## On your laptop

1. Download `freya-trained.gguf` from Google Drive into `freya-v1/training/freya-gguf/`.
2. Build it **alongside** your current model and compare (same command on Windows, macOS and Linux):
   ```
   npm run ollama:create -- --base training/freya-gguf/freya-trained.gguf --name freya-trained
   npm run eval:persona -- freya-trained
   npm run eval:persona -- freya
   ```
3. If it's better, make it the one the app uses:
   ```
   npm run ollama:create -- --base training/freya-gguf/freya-trained.gguf
   ```
   To go back: `npm run ollama:create`. If you trained the Qwen3 fallback base, add `--format chatml` to these commands.

Read the transcripts, not just the score. More good examples in `training/freya-examples.json` (a few hundred) is what improves a fine-tune most.